# DFU-RadialAdapter — GitHub-first, persistence-safe pilot
This notebook clones the validated source from GitHub. It refuses to train unless primary Drive, secondary Drive backup, and the `GITHUB_TOKEN` secret are available.

In [ ]:
# DFU-RadialAdapter GitHub-first pilot — one executable cell
import os, sys, shutil, subprocess, json
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=False)
if not Path('/content/drive/MyDrive').is_dir():
    raise RuntimeError('Google Drive is not mounted. No training started.')

token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Add GITHUB_TOKEN in Colab Secrets and enable notebook access. No training started.')
os.environ['GITHUB_TOKEN'] = token

subprocess.run([sys.executable,'-m','pip','install','-q',
    'timm>=1.0.9','kagglehub>=0.3','ImageHash>=4.3',
    'scikit-learn>=1.5','scipy>=1.13','matplotlib>=3.9',
    'pandas>=2.2','Pillow>=10.4','tabulate>=0.9'],check=True)

REPO='https://github.com/AzizulHakim00/DFU-ImageGuard.git'
SOURCE_BRANCH='radial-adapter-pilot-v1'
WORK=Path('/content/DFU-ImageGuard-radial')
if WORK.exists(): shutil.rmtree(WORK)
subprocess.run(['git','clone','--depth','1','--branch',SOURCE_BRANCH,REPO,str(WORK)],check=True)
os.chdir(WORK)
sys.path.insert(0,str(WORK))
for name in list(sys.modules):
    if name=='src' or name.startswith('src.'):
        del sys.modules[name]

from src.radial_pilot_runner import PilotSettings, run_radial_adapter_pilot

settings=PilotSettings(
    run_id='RADIAL_ADAPTER_PILOT_V1',
    drive_root='/content/drive/MyDrive/DFU-ImageGuard',
    secondary_drive_root='/content/drive/MyDrive/DFU-ImageGuard-Backup',
    seeds=(2026,2027),
    outer_fold=0,
    max_epochs=25,
    patience=7,
    batch_size=16,
    num_workers=2,
    github_export=True,
    github_export_required=True,
    github_branch='radial-pilot-results',
    github_chunk_full_checkpoints=True,
    github_chunk_bytes=48*1024*1024,
    github_export_after_each_trial=True,
    require_secondary_drive_backup=True,
)

result=run_radial_adapter_pilot(settings=settings,repository_dir=WORK)
print(json.dumps(result,indent=2,default=str))
